# virtualizarr smoke test

`virtualizarr` (v2 API) builds virtual Zarr views over archival files via
chunk manifests. Examples run fully offline: in-memory manifests, plus a
local HDF5 file virtualized through `HDFParser` + `obstore.store.LocalStore`.

Examples 3-5 need the HDF5 parser: `pip install "virtualizarr[hdf]"`.

## Example 1: `ChunkManifest` dict round-trip

In [ ]:
# Example 1: ChunkManifest — dict <-> array-backed manifest round-trip
# Source: adapted from the ChunkManifest docstring in virtualizarr/manifests/manifest.py (packaged wheel)
from virtualizarr.manifests import ChunkManifest

entries = {
    "0.0": {"path": "s3://bucket/foo.nc", "offset": 100, "length": 100},
    "0.1": {"path": "s3://bucket/foo.nc", "offset": 200, "length": 100},
    "1.0": {"path": "s3://bucket/foo.nc", "offset": 300, "length": 100},
    "1.1": {"path": "s3://bucket/foo.nc", "offset": 400, "length": 100},
}
manifest = ChunkManifest(entries=entries)

assert len(manifest) == 4
assert manifest.shape_chunk_grid == (2, 2)
assert manifest.dict() == entries  # lossless round-trip
print(f"manifest: {len(manifest)} chunks on grid {manifest.shape_chunk_grid}")
print(manifest.dict()["0.0"])


## Example 2: `ManifestArray` numpy-style concatenation

In [ ]:
# Example 2: ManifestArray — numpy-style concatenation without touching any bytes
# Source: adapted from virtualizarr/manifests/array_api.py np.concatenate dispatch (packaged wheel)
import numpy as np

from virtualizarr.manifests import ChunkManifest, ManifestArray
from virtualizarr.manifests.utils import create_v3_array_metadata

metadata = create_v3_array_metadata(
    shape=(4, 4), data_type=np.dtype("float32"), chunk_shape=(2, 2)
)

def marr_for(path):
    entries = {
        f"{i}.{j}": {"path": path, "offset": 16 * (2 * i + j), "length": 16}
        for i in range(2) for j in range(2)
    }
    return ManifestArray(metadata=metadata, chunkmanifest=ChunkManifest(entries=entries))

marr1, marr2 = marr_for("file:///data/a.nc"), marr_for("file:///data/b.nc")
combined = np.concatenate([marr1, marr2], axis=0)

assert isinstance(combined, ManifestArray)
assert combined.shape == (8, 4)
assert len(combined.manifest) == 8
print(f"concatenated: {marr1.shape} + {marr2.shape} -> {combined.shape}, {len(combined.manifest)} chunks")


## Example 3: `open_virtual_dataset` on a local HDF5 file

In [ ]:
# Example 3: open_virtual_dataset over a local HDF5 file (HDFParser + LocalStore)
# Source: adapted from virtualizarr/tests LocalStore + ObjectStoreRegistry pattern (packaged wheel)
import os
import tempfile

import h5py
import numpy as np
import xarray as xr
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import LocalStore

from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser

tmpdir = tempfile.mkdtemp()
h5_path = os.path.join(tmpdir, "sample.h5")
temperature = np.random.default_rng(seed=0).random((4, 6)).astype("float32")
with h5py.File(h5_path, "w") as f:
    f.create_dataset("temperature", data=temperature, chunks=(2, 3))

registry = ObjectStoreRegistry({"file://": LocalStore()})
vds = open_virtual_dataset(url=f"file://{h5_path}", registry=registry, parser=HDFParser())

assert isinstance(vds, xr.Dataset)
assert vds["temperature"].shape == (4, 6)
print(vds)


## Example 4: kerchunk references round-trip

(continues from Example 3's `vds` / `registry` / `tmpdir`)

In [ ]:
# Example 4: serialize to kerchunk refs and re-open them — full virtual round-trip
# Source: adapted from the to_kerchunk docstring in virtualizarr/accessor.py (packaged wheel)
import os

import ujson

from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import KerchunkJSONParser

refs = vds.vz.to_kerchunk(format="dict")
assert "refs" in refs and any(k.startswith("temperature/") for k in refs["refs"])

kerchunk_path = os.path.join(tmpdir, "refs.json")
with open(kerchunk_path, "w") as f:
    ujson.dump(refs, f)

roundtripped = open_virtual_dataset(
    url=f"file://{kerchunk_path}", registry=registry, parser=KerchunkJSONParser()
)
assert set(roundtripped.data_vars) == set(vds.data_vars)
assert roundtripped["temperature"].shape == vds["temperature"].shape
print("kerchunk round-trip preserved:", dict(roundtripped.sizes))


## Example 5: `loadable_variables` reads real bytes back

(continues from Example 3's file — data-integrity invariant through the manifest IO path)

In [ ]:
import numpy as np

from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser

loaded = open_virtual_dataset(
    url=f"file://{h5_path}",
    registry=registry,
    parser=HDFParser(),
    loadable_variables=["temperature"],
)

actual = loaded["temperature"].values
np.testing.assert_array_equal(actual, temperature)
print("byte-identical data read back through the manifest:", actual.shape, actual.dtype)